# 12 — BrainSMASH: GUSTO Longitudinal Pace vs AHBA Gene Expression Gradients

Tests whether the GUSTO longitudinal brain-pace pattern (PC1 loadings from delta-Z V1→V2) aligns with AHBA gene expression gradients (Deary et al. top-8k genes, HCP MMP1.0, 3 PCs), accounting for spatial autocorrelation via BrainSMASH.

**Inputs:**
- `GUSTO_LOADINGS_CSV` — GUSTO longitudinal PC1 loadings (`GUSTO_DeltaZ_PC1_Loadings.csv`, output of GUSTO PCA)
- `HCP_ATLAS` — HCP MMP1.0 parcellation (32k fs_LR)
- `DK_ATLAS` — DK aparc parcellation (32k fs_LR)
- `FSAVG5_LABEL_DIR` — fsaverage5 FreeSurfer label directory

**Output:** `brainsmash_results_gusto_longitudinal.csv` — r, uncorrected p, FDR-corrected p for each gene PC

**Pipeline:** AHBA → HCP parcels → DK parcels → bilateral BrainSMASH (68 cortical ROIs) → FDR (3 tests)

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
GENE_GRADIENTS_URL = (
    "https://raw.githubusercontent.com/richardajdear/AHBA_gradients/"
    "ab7939cf1811cba6296b882e35e60b09fed7d653/outputs/"
    "ahba_dme_hcp_top8kgenes_scores.csv"
)

HCP_ATLAS        = 'data/atlases/HCP_MMP1.32k_fs_LR.dlabel.nii'
DK_ATLAS         = 'data/atlases/fsaverage.aparc.32k_fs_LR.dlabel.nii'
FSAVG5_LABEL_DIR = '~/freesurfer/subjects/fsaverage5/label'

# GUSTO longitudinal PC1 loadings (output of GUSTO PCA on delta-Z V1→V2)
GUSTO_LOADINGS_CSV = 'data/external/GUSTO_DeltaZ_PC1_Loadings.csv'

OUTPUT_DIR = 'outputs/neuromaps'

N_PERM = 10000
SEED   = 42

GENE_PCS = ['PC1', 'PC2', 'PC3']

In [ ]:
import os
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FormatStrFormatter
import nilearn.datasets as nilearn_datasets
import nilearn.surface as nilearn_surface
from brainsmash.mapgen.base import Base
from statsmodels.stats.multitest import multipletests

os.makedirs(OUTPUT_DIR, exist_ok=True)

## Step 1 — Parcellate AHBA gene gradients into DK atlas

In [ ]:
df_genes   = pd.read_csv(GENE_GRADIENTS_URL)
atlas_data = nib.load(HCP_ATLAS).get_fdata()[0]

# Project each HCP parcel's gene scores to vertex space (bilateral)
vert_maps = {f'PC{i}': np.full(atlas_data.shape, np.nan) for i in range(1, 4)}
for _, row in df_genes.iterrows():
    combined = (atlas_data == row.id) | (atlas_data == (row.id + 180))
    for i in [1, 2, 3]:
        vert_maps[f'PC{i}'][combined] = row[f'C{i}']

# Average into DK parcels
dk_atlas  = nib.load(DK_ATLAS)
dk_data   = dk_atlas.get_fdata()[0]
dk_labels = list(dk_atlas.header.get_index_map(0).named_maps)[0].label_table

dk_results = []
for idx, label_obj in dk_labels.items():
    if label_obj.label in ['???', 'unknown', 'Medial_Wall', 'Background']:
        continue
    dk_mask = (dk_data == idx)
    if np.any(dk_mask):
        dk_results.append({
            'ROI': label_obj.label,
            'PC1': np.nanmean(vert_maps['PC1'][dk_mask]),
            'PC2': np.nanmean(vert_maps['PC2'][dk_mask]),
            'PC3': np.nanmean(vert_maps['PC3'][dk_mask]),
        })

df_gene_dk = pd.DataFrame(dk_results)
# Glasser label format (L_bankssts) → DK format (L.bankssts)
df_gene_dk['ROI']    = df_gene_dk['ROI'].apply(lambda n: n.replace('_', '.', 1))
df_gene_dk['hemi']   = np.where(df_gene_dk['ROI'].str.startswith('L.'), 'lh',
                        np.where(df_gene_dk['ROI'].str.startswith('R.'), 'rh', 'sub'))
df_gene_dk['Region'] = df_gene_dk['ROI'].str[2:]
df_gene_dk           = df_gene_dk[df_gene_dk['hemi'].isin(['lh', 'rh'])].copy()

print(f'Gene gradient parcels: {len(df_gene_dk)} DK ROIs')

## Step 2 — Build fsaverage5 parcel centroids for BrainSMASH distance matrix

In [ ]:
label_dir = os.path.expanduser(FSAVG5_LABEL_DIR)

fsavg      = nilearn_datasets.fetch_surf_fsaverage(mesh='fsaverage5')
lh_coords  = nilearn_surface.load_surf_mesh(fsavg['pial_left']).coordinates
rh_coords  = nilearn_surface.load_surf_mesh(fsavg['pial_right']).coordinates

dk_l, _, names_l = nib.freesurfer.read_annot(os.path.join(label_dir, 'lh.aparc.annot'))
dk_r, _, names_r = nib.freesurfer.read_annot(os.path.join(label_dir, 'rh.aparc.annot'))
names_l = [n.decode() if isinstance(n, bytes) else n for n in names_l]
names_r = [n.decode() if isinstance(n, bytes) else n for n in names_r]

DK_ORDER = [
    'bankssts', 'caudalanteriorcingulate', 'caudalmiddlefrontal', 'cuneus',
    'entorhinal', 'frontalpole', 'fusiform', 'inferiorparietal',
    'inferiortemporal', 'insula', 'isthmuscingulate', 'lateraloccipital',
    'lateralorbitofrontal', 'lingual', 'medialorbitofrontal', 'middletemporal',
    'paracentral', 'parahippocampal', 'parsopercularis', 'parsorbitalis',
    'parstriangularis', 'pericalcarine', 'postcentral', 'posteriorcingulate',
    'precentral', 'precuneus', 'rostralanteriorcingulate', 'rostralmiddlefrontal',
    'superiorfrontal', 'superiorparietal', 'superiortemporal', 'supramarginal',
    'temporalpole', 'transversetemporal',
]

def parcel_centroids(regions, surf_coords, annot_labels, annot_names):
    name_map = {name: i for i, name in enumerate(annot_names)}
    coords = []
    for region in regions:
        lab_idx = name_map.get(region)
        mask    = (annot_labels == lab_idx)
        coords.append(surf_coords[mask].mean(axis=0))
    return np.array(coords)

c_lh = parcel_centroids(DK_ORDER, lh_coords, dk_l, names_l)
c_rh = parcel_centroids(DK_ORDER, rh_coords, dk_r, names_r)
coords_bi = np.vstack([c_lh, c_rh])          # (68, 3)

print(f'Bilateral centroid matrix: {coords_bi.shape}')

## Step 3 — Load GUSTO longitudinal PC1 loadings

In [ ]:
# ABCD ROI code → DK parcel name (L./R. prefix)
ABCD_TO_DK = {
    'mr_y_smri__vol__dsk__bstmps__lh_sum': 'L.bankssts',
    'mr_y_smri__vol__dsk__cac__lh_sum':    'L.caudalanteriorcingulate',
    'mr_y_smri__vol__dsk__cmfrt__lh_sum':  'L.caudalmiddlefrontal',
    'mr_y_smri__vol__dsk__cn__lh_sum':     'L.cuneus',
    'mr_y_smri__vol__dsk__er__lh_sum':     'L.entorhinal',
    'mr_y_smri__vol__dsk__ff__lh_sum':     'L.fusiform',
    'mr_y_smri__vol__dsk__iprt__lh_sum':   'L.inferiorparietal',
    'mr_y_smri__vol__dsk__itmp__lh_sum':   'L.inferiortemporal',
    'mr_y_smri__vol__dsk__ic__lh_sum':     'L.isthmuscingulate',
    'mr_y_smri__vol__dsk__locc__lh_sum':   'L.lateraloccipital',
    'mr_y_smri__vol__dsk__lobfrt__lh_sum': 'L.lateralorbitofrontal',
    'mr_y_smri__vol__dsk__lg__lh_sum':     'L.lingual',
    'mr_y_smri__vol__dsk__mobfrt__lh_sum': 'L.medialorbitofrontal',
    'mr_y_smri__vol__dsk__mtmp__lh_sum':   'L.middletemporal',
    'mr_y_smri__vol__dsk__ph__lh_sum':     'L.parahippocampal',
    'mr_y_smri__vol__dsk__pactr__lh_sum':  'L.paracentral',
    'mr_y_smri__vol__dsk__pop__lh_sum':    'L.parsopercularis',
    'mr_y_smri__vol__dsk__pob__lh_sum':    'L.parsorbitalis',
    'mr_y_smri__vol__dsk__ptg__lh_sum':    'L.parstriangularis',
    'mr_y_smri__vol__dsk__pcc__lh_sum':    'L.pericalcarine',
    'mr_y_smri__vol__dsk__poctr__lh_sum':  'L.postcentral',
    'mr_y_smri__vol__dsk__pcg__lh_sum':    'L.posteriorcingulate',
    'mr_y_smri__vol__dsk__prctr__lh_sum':  'L.precentral',
    'mr_y_smri__vol__dsk__prcn__lh_sum':   'L.precuneus',
    'mr_y_smri__vol__dsk__rac__lh_sum':    'L.rostralanteriorcingulate',
    'mr_y_smri__vol__dsk__rmfrt__lh_sum':  'L.rostralmiddlefrontal',
    'mr_y_smri__vol__dsk__sfrt__lh_sum':   'L.superiorfrontal',
    'mr_y_smri__vol__dsk__sprt__lh_sum':   'L.superiorparietal',
    'mr_y_smri__vol__dsk__stmp__lh_sum':   'L.superiortemporal',
    'mr_y_smri__vol__dsk__sm__lh_sum':     'L.supramarginal',
    'mr_y_smri__vol__dsk__pfrt__lh_sum':   'L.frontalpole',
    'mr_y_smri__vol__dsk__ptmp__lh_sum':   'L.temporalpole',
    'mr_y_smri__vol__dsk__ttmp__lh_sum':   'L.transversetemporal',
    'mr_y_smri__vol__dsk__ins__lh_sum':    'L.insula',
    'mr_y_smri__vol__dsk__bstmps__rh_sum': 'R.bankssts',
    'mr_y_smri__vol__dsk__cac__rh_sum':    'R.caudalanteriorcingulate',
    'mr_y_smri__vol__dsk__cmfrt__rh_sum':  'R.caudalmiddlefrontal',
    'mr_y_smri__vol__dsk__cn__rh_sum':     'R.cuneus',
    'mr_y_smri__vol__dsk__er__rh_sum':     'R.entorhinal',
    'mr_y_smri__vol__dsk__ff__rh_sum':     'R.fusiform',
    'mr_y_smri__vol__dsk__iprt__rh_sum':   'R.inferiorparietal',
    'mr_y_smri__vol__dsk__itmp__rh_sum':   'R.inferiortemporal',
    'mr_y_smri__vol__dsk__ic__rh_sum':     'R.isthmuscingulate',
    'mr_y_smri__vol__dsk__locc__rh_sum':   'R.lateraloccipital',
    'mr_y_smri__vol__dsk__lobfrt__rh_sum': 'R.lateralorbitofrontal',
    'mr_y_smri__vol__dsk__lg__rh_sum':     'R.lingual',
    'mr_y_smri__vol__dsk__mobfrt__rh_sum': 'R.medialorbitofrontal',
    'mr_y_smri__vol__dsk__mtmp__rh_sum':   'R.middletemporal',
    'mr_y_smri__vol__dsk__ph__rh_sum':     'R.parahippocampal',
    'mr_y_smri__vol__dsk__pactr__rh_sum':  'R.paracentral',
    'mr_y_smri__vol__dsk__pop__rh_sum':    'R.parsopercularis',
    'mr_y_smri__vol__dsk__pob__rh_sum':    'R.parsorbitalis',
    'mr_y_smri__vol__dsk__ptg__rh_sum':    'R.parstriangularis',
    'mr_y_smri__vol__dsk__pcc__rh_sum':    'R.pericalcarine',
    'mr_y_smri__vol__dsk__poctr__rh_sum':  'R.postcentral',
    'mr_y_smri__vol__dsk__pcg__rh_sum':    'R.posteriorcingulate',
    'mr_y_smri__vol__dsk__prctr__rh_sum':  'R.precentral',
    'mr_y_smri__vol__dsk__prcn__rh_sum':   'R.precuneus',
    'mr_y_smri__vol__dsk__rac__rh_sum':    'R.rostralanteriorcingulate',
    'mr_y_smri__vol__dsk__rmfrt__rh_sum':  'R.rostralmiddlefrontal',
    'mr_y_smri__vol__dsk__sfrt__rh_sum':   'R.superiorfrontal',
    'mr_y_smri__vol__dsk__sprt__rh_sum':   'R.superiorparietal',
    'mr_y_smri__vol__dsk__stmp__rh_sum':   'R.superiortemporal',
    'mr_y_smri__vol__dsk__sm__rh_sum':     'R.supramarginal',
    'mr_y_smri__vol__dsk__pfrt__rh_sum':   'R.frontalpole',
    'mr_y_smri__vol__dsk__ptmp__rh_sum':   'R.temporalpole',
    'mr_y_smri__vol__dsk__ttmp__rh_sum':   'R.transversetemporal',
    'mr_y_smri__vol__dsk__ins__rh_sum':    'R.insula',
}

df_gusto = pd.read_csv(GUSTO_LOADINGS_CSV)
df_gusto = df_gusto.rename(columns={df_gusto.columns[0]: 'ROI'})
df_gusto['ROI'] = df_gusto['ROI'].replace(ABCD_TO_DK)

# Keep cortical ROIs only
df_gusto = df_gusto[df_gusto['ROI'].str.match(r'^[LR]\.')].copy()
df_gusto['hemi']   = np.where(df_gusto['ROI'].str.startswith('L.'), 'lh', 'rh')
df_gusto['Region'] = df_gusto['ROI'].str[2:]

# Auto-detect the loading column
loading_col = [c for c in df_gusto.columns if 'Loading' in c][0]
print(f'Loading column: {loading_col}')

gusto_lh = df_gusto[df_gusto['hemi'] == 'lh'].set_index('Region').reindex(DK_ORDER)
gusto_rh = df_gusto[df_gusto['hemi'] == 'rh'].set_index('Region').reindex(DK_ORDER)
print(f'GUSTO longitudinal ROIs: {len(gusto_lh)} LH + {len(gusto_rh)} RH')

## Step 4 — BrainSMASH: GUSTO pace loadings vs each gene gradient PC

In [ ]:
def run_brainsmash(x, y, coords, n_perm=10000, seed=42):
    D          = np.linalg.norm(coords[:, None, :] - coords[None, :, :], axis=-1)
    gen        = Base(x=y, D=D, seed=seed, resample=True)
    surrogates = gen(n=n_perm)
    r_obs      = np.corrcoef(x, y)[0, 1]
    r_nulls    = np.array([np.corrcoef(x, surrogates[i])[0, 1] for i in range(n_perm)])
    p_val      = (np.sum(np.abs(r_nulls) >= np.abs(r_obs)) + 1) / (n_perm + 1)
    return float(r_obs), float(p_val)


gene_lh = df_gene_dk[df_gene_dk['hemi'] == 'lh'].set_index('Region').reindex(DK_ORDER)
gene_rh = df_gene_dk[df_gene_dk['hemi'] == 'rh'].set_index('Region').reindex(DK_ORDER)

# GUSTO pace loadings (bilateral vector)
x_bi = np.concatenate([
    gusto_lh[loading_col].astype(float).values,
    gusto_rh[loading_col].astype(float).values,
])

results = []
for pc in GENE_PCS:
    y_bi = np.concatenate([
        gene_lh[pc].astype(float).values,
        gene_rh[pc].astype(float).values,
    ])
    r_obs, p_val = run_brainsmash(x_bi, y_bi, coords_bi, n_perm=N_PERM, seed=SEED)
    results.append({'Gene_PC': pc, 'R_Correlation': r_obs, 'P_Uncorrected': p_val})
    print(f'  {pc}: r = {r_obs:.3f}  p = {p_val:.5f}')

print('BrainSMASH complete.')

## Step 5 — FDR correction and save

In [ ]:
df_results = pd.DataFrame(results)

_, p_fdr, _, _ = multipletests(df_results['P_Uncorrected'], method='fdr_bh')
df_results['P_FDR']       = p_fdr
df_results['Significant'] = np.where(df_results['P_FDR'] < 0.05, 'Yes', 'No')

print(df_results.to_string(index=False))

out_path = os.path.join(OUTPUT_DIR, 'brainsmash_results_gusto_longitudinal.csv')
df_results.to_csv(out_path, index=False)
print(f'\nSaved: {out_path}')